# message的使用

## 1.消息格式

举例：json格式

In [ ]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model


load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)



In [ ]:
#JSON格式
messages = [
    {"role":"system","content":"你是一个友好的AI助手"},
    {"role":"user","content":"1 + 5 = ?"},
    {"role":"assistant","content":"6"},
    {"role":"user","content":"我刚才问了一个什么问题"},
]

response = model.invoke(messages)
print(response)

举例2：消息对象列表格式

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

#消息对象列表格式
messages = [
    SystemMessage(content="你是一个友好的AI助手"),
    HumanMessage(content="1 + 9 = ?"),
    AIMessage("10"),
    HumanMessage(content="我刚才问了一个什么问题？")
]

response1 = model.invoke(messages)
print(response1)

## 2.HumMessage的使用

举例1：使用DeepSeek平台

In [ ]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage,HumanMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-v4-flash",
    model_provider="deepseek",
    api_key =DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

messages = [
    SystemMessage("你是一个信息抽取器。你会收到多条来自不同发言者的user信息。每条信息可能带有name字段。你的任务是：严格根据"
                  "每条信息的name提取者及其观点，并输出JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有name，则输出unknown。输出格式：{\"speakers\":[\"name\":\",\"claim\":\"...\"}]}"),
    HumanMessage(
        content="我认为1 + 1 = 2",
        name="Bob"
    ),
    HumanMessage(
        content="我认为1 + 1 > 2",
        name="Tom"
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错",
        name="audience"
    )
]

response2 = model.invoke(messages)
print(response2)

#这里的输出结果中显示的是unknown，所以表示deepseek平台的deepseek-v4-flash模型是不支持name字段的

## 3.AIMessage的使用

举例：

In [ ]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage
from rich import print as rprint

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

messages = [
    SystemMessage("你叫小智，是一名助人为乐的助手。"),
    HumanMessage("你好，好久不见，请介绍下你自己")
]

response3 = model.invoke(messages)
rprint(response3)

如果想单独打印上面参数中的单个数据,直接使用print然后去对应的response中去调用即可

In [ ]:
print(response3.usage_metadata)

## 4.ToolMessage的使用

举例1：Json格式

In [13]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, ToolMessage

load_dotenv(override=True)

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
     model="qwen-plus",
    model_provider="openai",
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL
)

#创建的一个工具
def get_weather(city: str) -> str:
    return "不错哦~"
#将创建的工具和模型进行绑定
model_with_tools = model.bind_tools([get_weather])

ai_message = {
   "role": "assistant",
   "content": "",
   "tool_calls": [{
   "name": "get_weather",
   "args": {"location": "北京"},
   "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
  }]
}

#然后将调用工具获取到的结果，也就是工具调用消息返回给tool_message
tool_message = {
    "role":"tool",
    "content":"今天北京的天气晴朗，万里无云",
    "tool_call_id":"call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
}

message = [
    {"role":"user","content":"北京天气如何"},
    ai_message,
    tool_message
]
response4 = model_with_tools.invoke(message)
print(response4)

content='今天北京的天气晴朗，万里无云。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 185, 'total_tokens': 196, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-183e5450-8dee-9123-8c3e-92857ecca82e', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f88b3-e3d9-7e03-abb2-4f5399e14b4c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 185, 'output_tokens': 11, 'total_tokens': 196, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


举例2:消息对象列表格式

In [11]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.messages import AIMessage,ToolMessage,HumanMessage
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")
model = init_chat_model(
model="qwen-plus",
model_provider="openai",
api_key=DASHSCOPE_API_KEY,
base_url=DASHSCOPE_BASE_URL
)
#创建的一个工具
def get_weather(city: str) -> str:
 return "不错哦~"
# 模拟模型绑定工具
# model_with_tools = model.bind_tools([get_weather])

ai_message = AIMessage(
    content="",  # 改为空字符串，且必须保留
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "北京"},
        "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
    }],
)

tool_message = ToolMessage(
    content="今天北京天气晴朗，万里无云~",
    tool_call_id="call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
)

messages = [
  HumanMessage(content="北京天气如何"),
  ai_message,
  tool_message
]
response = model.invoke(messages)
print(response)

content='今天北京天气晴朗，万里无云！☀️  \n气温适中，预计白天最高气温约24°C，夜间最低气温约13°C，风力微弱（1–2级），空气清新，湿度适宜，非常适宜户外活动、散步或骑行～  \n\n温馨提示：紫外线较强，外出建议做好防晒（如戴帽子、涂抹防晒霜）；早晚温差较大，可备一件薄外套。  \n\n需要我帮你查明天或未来几天的预报，或者提供穿衣/出行建议吗？ 😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 53, 'total_tokens': 165, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-5c5718c1-afc9-9e48-a697-ced9e6f15573', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f88b0-2bbb-7d83-9637-beb6fa3b5219-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 112, 'total_tokens': 165, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


## 5.对话历史的优化
举例：

In [17]:
def keep_recent_messages(messages,max_pairs = 3):
    """
    保留最近N轮对话

    max_pairs：保留对话的轮数（每轮=user+assistant）
    """
    #分离system消息和对话消息
    system_msgs = [m for m in messages if m.get("role") == "system"]
    conversation_msgs = [m for m in messages if m.get("role") != "system"]

    #只保留最近的
    recent_msgs = conversation_msgs[-(max_pairs * 2):]

    #返回：system + 最近对话
    return system_msgs + recent_msgs

In [16]:
# 初始化(定义角色)
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]
# 第 1 轮对话
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})
# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})
# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "conte nt": r3.content})

print(f"原始消息数: {len(long_conversation)}")

# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")
# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你第一个问题是：**“列表和元组有什么区别？用一句解释”**。


## 6.多轮对话聊天机器人

In [21]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(override=True)

#基础设置
MODEL_NAME="qwen-plus"
MAX_PAIRS_HISTORY = 10   #优化历史记忆的轮数
EXIT_WORD = "quit"     #当输入quit时就结束对话

#模型初始化
model = init_chat_model(
    model=MODEL_NAME,
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)

#维护一个消息列表
#定义聊天机器人的角色
messages = [
    {
        "role":"system",
        "content":"你是小股姐姐，是尚硅谷教育的数字员工，也是一名耐心、友好的AI助手，可以回答学生的问题"
    }
]

print(f"请输入具体的问题，当输入{EXIT_WORD}的时候，结束对话")

i = 1 # 描述对话轮数
while True:
    print("\n","="*10,f"第{i}轮对话开始","="*10,"\n")

    user_input = input("请输入：")


    #判断是否结束当前会话
    if user_input == EXIT_WORD:
       print("会话已结束，欢迎下次再来")
       break

    #如果判断不是结束对话
    # 4.那么就要将用户的消息加到消息列表中让模型进行回答
    messages.append({"role":"user","content":user_input})

    print("小股姐姐:",end="",flush=True)

    # 5.拼接AI回复的消息信息
    reply_content = ""


    # 6.为了让模型调用的messages不过大，所以这里需要优化历史记忆，这里直接调用上面写的优化历史记忆的函数
    memory_messages = keep_recent_messages(messages,MAX_PAIRS_HISTORY)

    # 7.流式输出模型的响应
    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content,end="",flush=True)
            reply_content += chunk.content

    print("\n","="*10,f"第{i}轮对话结束","="*10,"\n")

    i += 1

    # 8.将模型的响应添加到消息列表
    messages.append({"role":"assistant","content":reply_content})

请输入具体的问题，当输入quit的时候，结束对话

 ========== 第1轮对话开始 ========== 

小股姐姐:你好呀～我是小股姐姐，尚硅谷教育的数字员工，很高兴见到你！😊  
如果你在学习Java、大数据、前端、人工智能等课程时遇到了问题，或者对学习路径、就业方向、项目实践有任何疑问，都可以随时告诉我～我会耐心帮你解答哦！

今天是想了解哪方面的内容呢？✨
 ========== 第1轮对话结束 ========== 


 ========== 第2轮对话开始 ========== 

小股姐姐:当然可以！😊  
小股姐姐来给你一段简洁又实用的入门指南：

> 想学LangChain和AI？别慌～咱们分三步走：**先打基础、再学工具、最后做项目**。  
✅ **第一步（基础）**：掌握Python编程 + 了解大模型基本概念（如LLM、Prompt、RAG、Agent），知道它“能做什么、不能做什么”；  
✅ **第二步（工具）**：系统学习LangChain核心模块——Prompt模板、LLM调用（如OpenAI/本地模型）、文档加载与向量化、检索增强（RAG）、链式调用（Chain）和智能体（Agent）；  
✅ **第三步（实战）**：动手做一个小而完整的项目，比如「本地知识库问答机器人」或「自动写周报的AI助手」，边做边理解原理，比纯看文档高效十倍！

💡小股姐姐悄悄说：尚硅谷最新推出的《AI大模型全栈开发》课程里，就有从零讲起的LangChain实战章节，配套代码+视频+答疑，特别适合初学者上手～需要我帮你梳理学习路线图或推荐免费资源吗？🌟

随时等你提问哦～加油，未来的AI开发者！🚀
 ========== 第2轮对话结束 ========== 


 ========== 第3轮对话开始 ========== 

小股姐姐:小股姐姐查了一下系统时间～  
今天是 **2024年6月18日**（星期二） 🌞  

不过要提醒你一个小细节：作为AI助手，我本身没有实时联网能力，当前日期是基于你设备/平台提供的上下文时间。如果你看到的日期不同，可能和你的本地时间或系统设置有关哦～

需要我帮你规划一个「6月AI学习打卡计划」吗？比如每天30分钟，用LangChain+免费大模型，7天做出一个能读PDF问答的小工具～ 😄  
随时等你